# K-means en Google Colab — seq + OpenMP + CUDA

**Proyecto Semestral — Introducción a la Computación Paralela**

Este notebook compila y mide las **tres** versiones de K-means
(secuencial, OpenMP y CUDA) en la **misma máquina de Colab**, que provee
una GPU NVIDIA real con `nvcc` — sin instalar nada en tu computador.

## Antes de empezar (IMPORTANTE)

Activa la GPU: menú **Entorno de ejecución → Cambiar tipo de entorno de
ejecución → Acelerador por hardware: GPU → Guardar**.

Luego ejecuta las celdas en orden (o **Entorno de ejecución → Ejecutar
todas**). El dataset real `punto0` viene dentro del repositorio, así que
no hay que subir archivos.

## 1. Verificar GPU y toolchain

Conviene anotar el modelo de GPU y el número de vCPUs de la máquina.

In [ ]:
print('===== GPU =====')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo 'SIN GPU: activa el acelerador (ver arriba)'
print('\n===== nvcc (CUDA) =====')
!nvcc --version | grep release || echo 'nvcc no disponible'
print('\n===== CPU =====')
!nproc
!gcc --version | head -1

## 2. Clonar el repositorio

Trae el código, los scripts y el dataset real `punto0`. Si ya estaba
clonado, hace `git pull` para traer la última versión.

In [ ]:
import os
REPO   = 'https://github.com/JackFeels/kmeans-paralelo.git'
TARGET = '/content/kmeans-paralelo'
if not os.path.exists(TARGET):
    !git clone {REPO} {TARGET}
else:
    !cd {TARGET} && git pull
os.chdir(TARGET)
print('cwd:', os.getcwd())
!ls

## 3. Compilar las tres versiones

El `Makefile` detecta Linux y usa `gcc -fopenmp` (seq/omp) y `nvcc` (CUDA).

In [ ]:
!make clean
print('--- CPU (seq + omp) ---')
!make cpu
print('\n--- CUDA ---')
!make cuda
print('\n--- Binarios ---')
!ls -la kmeans_seq kmeans_omp kmeans_cuda

## 4. Generar datasets sintéticos

Semilla 7 = idénticos a los locales. `punto0` ya vino con el repo.

In [ ]:
!python3 scripts/gen_dataset.py 1000   2  3  data/small_2d.txt
!python3 scripts/gen_dataset.py 10000  10 5  data/medium_10d.txt
!python3 scripts/gen_dataset.py 100000 50 10 data/large_50d.txt
print()
!ls -lh data/

## 5. Smoke test

Una corrida de cada versión sobre el dataset real, para confirmar que las
tres funcionan antes del barrido largo.

In [ ]:
DS = 'data/punto0_kmeans.txt'
!./kmeans_seq {DS} 5 100
!OMP_NUM_THREADS=2 ./kmeans_omp {DS} 5 100
!./kmeans_cuda {DS} 5 100

## 6. Detectar núcleos

> **Nota.** Colab (gratis) suele dar **2 vCPUs**, así que la *curva de
> escalabilidad de OpenMP* aquí es corta. Eso **no** afecta la comparación
> seq/OMP/CUDA, que es el objetivo y se mide bien en esta misma máquina con
> GPU. La escalabilidad de OpenMP con muchos núcleos se observa mejor en un
> CPU multinúcleo (ver run_all.sh / run_all.ps1).

In [ ]:
import multiprocessing, os
ncpu = multiprocessing.cpu_count()
ths, t = [], 1
while t <= ncpu:
    ths.append(t); t *= 2
if ncpu not in ths: ths.append(ncpu)
ths.append(min(ncpu * 2, ncpu + 8))
ths = sorted(set(ths))
THREADS_LIST = ' '.join(map(str, ths))
os.environ['THREADS_LIST'] = THREADS_LIST
print(f'vCPUs: {ncpu}  ->  THREADS_LIST = [{THREADS_LIST}]')

## 7. Benchmark completo

Mide seq + OpenMP + CUDA para cada dataset y cada K, y vuelca **un solo**
`results/benchmark.csv`.

⚠️ **Tiempo de ejecución.** `K=200` es realista pero pesado (decenas de
segundos por corrida en CPU). El barrido completo con `REPS=3` puede
tardar **20–40 min**. Mantén la pestaña activa para que Colab no te
desconecte.

**Recomendación:** corre primero la *prueba rápida* (celda de abajo con
`REPS=1`, `K_LIST='3 5'`) para confirmar que CUDA mide bien; luego la
corrida completa.

In [ ]:
# --- PRUEBA RÁPIDA (descomenta para validar en ~1 min) ---
# REPS, K_LIST = 1, '3 5'

# --- CORRIDA COMPLETA (por defecto) ---
REPS, K_LIST = 3, '3 5 10 200'

import os
os.environ['REPS']     = str(REPS)
os.environ['MAX_ITER'] = '100'
os.environ['K_LIST']   = K_LIST
print(f'REPS={REPS}  K_LIST=[{K_LIST}]  THREADS_LIST=[{os.environ["THREADS_LIST"]}]\n')
!bash scripts/run_benchmark.sh

In [ ]:
# Verificar que quedaron las tres versiones en el CSV
import csv, os
from collections import Counter
rows = list(csv.DictReader(open('results/benchmark.csv')))
print('Filas:', len(rows))
print('Por version:', dict(Counter(r['version'] for r in rows)))
print('Datasets   :', sorted({os.path.basename(r['dataset']) for r in rows}))
assert 'cuda' in {r['version'] for r in rows}, 'No hay filas CUDA: revisa el paso 3 (make cuda).'

## 8. Descargar resultados

Baja `benchmark.csv` con los tiempos de las tres versiones (secuencial,
OpenMP y CUDA) medidos en esta misma máquina.

In [ ]:
from google.colab import files
files.download('results/benchmark.csv')